# 316 — Orchestrator & Direct API Usage

Demonstrates the multi-agent pipeline (Orchestrator → ComplianceAgent + InvestigationAgent)
via direct Python calls. The interactive chat UI has moved to  (Streamlit).

**Pre-loaded example questions** use real entity IDs from the data.

In [1]:
%run 311_agent_setup.ipynb

/opt/anaconda3/envs/graphrag-finserv/lib/python3.11/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Environment loaded.


2026-03-09 16:32:05,203 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io


Connected to Neo4j.

Node counts in graph:
  {'labels': {'CommercialSecured': 2, 'Federal': 1, 'Address': 609, 'Residential': 582, 'Corporate': 31, 'Assessment': 10, 'Collateral': 466, 'ResidentialSecured': 464, 'Industry': 14, 'National': 5, 'PersonalTransaction': 610, 'Threshold': 157, 'CorporateCurrent': 6, 'Borrower': 628, 'BusinessTransaction': 175, 'SpecialAdminRegion': 1, 'BankAccount': 791, 'Chunk': 205, 'Section': 118, 'Jurisdiction': 7, 'Commercial': 27, 'Requirement': 194, 'Transaction': 173, 'LoanApplication': 466, 'Individual': 597, 'Finding': 40, 'Regulation': 3, 'ReasoningStep': 55, 'Officer': 19}}
  ✓  Borrower: 628
  ✓  LoanApplication: 466
  ✓  BankAccount: 791
  ✓  Transaction: 173
  ✓  Regulation: 3
  ✓  Section: 118
  ✓  Requirement: 194
  ✓  Threshold: 157
  ✓  Chunk: 205
Anthropic client ready. Model: claude-sonnet-4-6
OpenAI client ready.
Tool registry: 2 Neo4j MCP + 5 FastMCP = 7 total
  read-neo4j-cypher
  write-neo4j-cypher
  traverse_compliance_path
  retrie

In [2]:
from src.agent.orchestrator import Orchestrator

orchestrator = Orchestrator(tools=TOOLS, execute_tool_fn=execute_tool)
print("Orchestrator ready.")

2026-03-09 16:32:06,497 [INFO] execute_tool: Tool: read-neo4j-cypher | inputs: ['query']
2026-03-09 16:32:06,534 [INFO] src.agent.orchestrator: Graph regulations found: ['APG-223', 'APS-112', 'APS-220']


Orchestrator ready.


## Direct API usage

Run questions programmatically and inspect the structured .

In [5]:
resp = orchestrator.run('Find all transaction structuring patterns.')
print(f'Verdict:    {resp.verdict}')
print(f'Confidence: {resp.confidence}')
print(f'Findings:   {len(resp.findings)}')
print(f'Cypher:     {len(resp.cypher_used)} queries')
print(f'Answer (first 500 chars):{resp.answer[:500]}')

2026-03-09 16:32:22,126 [INFO] src.agent.orchestrator: [938d9b87] Orchestrator: Find all transaction structuring patterns.
2026-03-09 16:32:23,646 [INFO] src.agent.orchestrator: [938d9b87] Routing: {'intents': ['anomaly', 'investigation'], 'entity_ids': [], 'entity_types': ['Transaction'], 'regulations': [], 'run_anomaly_check': True, 'needs_compliance_agent': False, 'needs_investigation_agent': True}
2026-03-09 16:32:23,648 [INFO] src.agent.investigation_agent: InvestigationAgent: Find all transaction structuring patterns.
2026-03-09 16:32:27,264 [INFO] src.agent.investigation_agent: Tool: detect_graph_anomalies(['pattern_name'])
2026-03-09 16:32:27,267 [INFO] execute_tool: Tool: detect_graph_anomalies | inputs: ['pattern_name']
2026-03-09 16:32:27,468 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io
2026-03-09 16:32:27,501 [INFO] src.graph.connection: Neo4j connection closed.
2026-03-09 16:32:27,501 [INFO] src.agent.investigation_agent: Tool: r

Verdict:    INFORMATIONAL
Confidence: 0.5
Findings:   6
Cypher:     7 queries
Answer (first 500 chars):## Synthesis: Transaction Structuring Patterns

### Direct Answer

Analysis of the transaction data reveals **systematic structuring (smurfing) activity across three borrower accounts**, with all 19 transactions deliberately kept below the AUD 10,000 reporting threshold — a hallmark of AUSTRAC threshold avoidance. The pattern is compounded by unknown fund sources, varied transaction descriptions designed to obscure intent, and a strong likelihood that structured deposits are being used to artifi


## Inspect findings

In [6]:
for f in resp.findings:
    print(f'[{f["severity"]}] {f["description"]}')

[HIGH] [HIGH] All 19 transactions are flagged_suspicious=true and sub-$10,000, consistent with AUSTRAC threshold avoidance
[HIGH] [HIGH] BRW-0602 (Hassan Martin) carries a pre-existing HIGH risk_rating with zero declared annual revenue, yet receives AUD 54,558 in 9 days
[HIGH] [HIGH] Source accounts are unresolved across all three clusters — funds origin is unknown/anonymous
[MEDIUM] [MEDIUM] All three borrowers have active loan applications under review (AUD 450K–600K), raising the possibility that structured deposits are being used to demonstrate false serviceability
[MEDIUM] [MEDIUM] Descriptions are deliberately varied within each cluster (Gift / Family support / Personal / Loan repayment) — a classic obfuscation technique
[LOW] [LOW] All three borrowers are in JUR-AU-FED (low AML risk jurisdiction), but structuring is a domestic pattern that does not require offshore exposure


## Inspect routing decision

In [7]:
import json
print(json.dumps(resp.routing, indent=2))

{
  "intents": [
    "anomaly",
    "investigation"
  ],
  "entity_ids": [],
  "entity_types": [
    "Transaction"
  ],
  "regulations": [],
  "run_anomaly_check": true,
  "needs_compliance_agent": false,
  "needs_investigation_agent": true
}
